In [0]:
# ==========================================================
# NOTEBOOK : 05_Gold
# PURPOSE  : Create Gold Layer (Dimensions & Fact Table)
# ==========================================================

from pyspark.sql import functions as F

print("Gold Notebook Started")

# ==========================================================
# STORAGE CONFIGURATION
# ==========================================================

storage_account = "retailadls67"
access_key = ""

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    access_key
)

# ==========================================================
# SILVER INPUT PATHS
# ==========================================================

SILVER_PRODUCTS = f"abfss://silver@{storage_account}.dfs.core.windows.net/products"
SILVER_CUSTOMERS = f"abfss://silver@{storage_account}.dfs.core.windows.net/customers"
SILVER_ORDERS = f"abfss://silver@{storage_account}.dfs.core.windows.net/orders"

# ==========================================================
# GOLD OUTPUT PATHS
# ==========================================================

GOLD_DIM_PRODUCT = f"abfss://gold@{storage_account}.dfs.core.windows.net/dim_product"
GOLD_DIM_CUSTOMER = f"abfss://gold@{storage_account}.dfs.core.windows.net/dim_customer"
GOLD_FACT_SALES = f"abfss://gold@{storage_account}.dfs.core.windows.net/fact_sales"

# ==========================================================
# READ SILVER DELTA
# ==========================================================

products_df = spark.read.format("delta").load(SILVER_PRODUCTS)
customers_df = spark.read.format("delta").load(SILVER_CUSTOMERS)
orders_df = spark.read.format("delta").load(SILVER_ORDERS)

# ==========================================================
# DIM PRODUCT
# ==========================================================

dim_product = products_df.select(
    "ProductID",
    "ProductName",
    "Brand",
    "Category",
    "SubCategory",
    "SellingPrice",
    "Status"
).dropDuplicates()

# ==========================================================
# DIM CUSTOMER
# ==========================================================

dim_customer = customers_df.select(
    "CustomerID",
    "FirstName",
    "LastName",
    "Gender",
    "City",
    "State",
    "CustomerSegment",
    "PreferredCategory"
).dropDuplicates()

# ==========================================================
# FACT SALES
# ==========================================================

fact_sales = (
    orders_df
    .join(dim_customer, "CustomerID", "left")
    .join(dim_product, "ProductID", "left")
    .select(
        "OrderID",
        "CustomerID",
        "ProductID",
        "OrderDate",
        "Quantity",
        "Discount",
        "TotalAmount",
        "PaymentMethod",
        "StoreID",
        "OrderStatus"
    )
)

# ==========================================================
# WRITE GOLD DELTA
# ==========================================================

dim_product.write \
    .format("delta") \
    .mode("overwrite") \
    .save(GOLD_DIM_PRODUCT)

dim_customer.write \
    .format("delta") \
    .mode("overwrite") \
    .save(GOLD_DIM_CUSTOMER)

fact_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .save(GOLD_FACT_SALES)

# ==========================================================
# VALIDATION
# ==========================================================

print("=" * 60)
print("GOLD LAYER CREATED SUCCESSFULLY")
print("=" * 60)

print("Dim Product  :", dim_product.count())
print("Dim Customer :", dim_customer.count())
print("Fact Sales   :", fact_sales.count())

dim_product.show(5)
dim_customer.show(5)
fact_sales.show(5)

In [0]:
jdbcHostname = "retail-sql-server-67.database.windows.net"
jdbcPort = 1433
jdbcDatabase = "RetailDB"

jdbcUrl = (
    f"jdbc:sqlserver://{jdbcHostname}:{jdbcPort};"
    f"database={jdbcDatabase};"
    "encrypt=true;"
    "trustServerCertificate=false;"
    "hostNameInCertificate=*.database.windows.net;"
    "loginTimeout=30;"
)

connectionProperties = {
    "user": "sqladmin",
    "password": "Murali@2004",
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}


dim_customer.write \
.mode("overwrite") \
.jdbc(
    url=jdbcUrl,
    table="gold.dim_customer",
    properties=connectionProperties
)

dim_product.write \
.mode("overwrite") \
.jdbc(
    url=jdbcUrl,
    table="gold.dim_product",
    properties=connectionProperties
)

fact_sales.write \
.mode("overwrite") \
.jdbc(
    url=jdbcUrl,
    table="gold.fact_sales",
    properties=connectionProperties
)